1. who are our best customers with full profile?

In [0]:
%sql
-- ================================================
-- MV 1: Customer 360 View
-- Who are our best customers with full profile?
-- ================================================
CREATE OR REPLACE MATERIALIZED VIEW streaming_project.ecommerce.mv_customer_360
AS
SELECT
    c.customer_id,
    c.full_name,
    c.email,
    c.city,
    c.is_valid_email,
    r.total_orders,
    r.total_items_bought,
    r.total_revenue,
    r.avg_order_value,
    r.last_order_date,
    CASE
        WHEN r.total_revenue > 500  THEN 'Platinum'
        WHEN r.total_revenue > 200  THEN 'Gold'
        WHEN r.total_revenue > 100  THEN 'Silver'
        ELSE 'Bronze'
    END AS customer_tier
FROM streaming_project.ecommerce.silver_customers c
LEFT JOIN streaming_project.ecommerce.gold_customer_revenue r
    ON c.customer_id = r.customer_id
ORDER BY r.total_revenue DESC;

result
The operation was successfully executed.


2. Which products are driving the most revenue?

In [0]:
%sql
-- ================================================
-- MV 2: Product Performance View
-- Which products are driving the most revenue?
-- ================================================
CREATE OR REPLACE MATERIALIZED VIEW streaming_project.ecommerce.mv_product_performance
AS
SELECT
    p.product_id,
    p.title,
    p.category,
    p.price,
    p.price_category,
    p.rating_rate,
    p.rating_category,
    pp.total_quantity_sold,
    pp.total_orders,
    pp.unique_customers,
    pp.total_revenue,
    RANK() OVER (ORDER BY pp.total_revenue DESC)        AS revenue_rank,
    RANK() OVER (ORDER BY pp.total_quantity_sold DESC)  AS popularity_rank,
    RANK() OVER (
        PARTITION BY p.category
        ORDER BY pp.total_revenue DESC
    ) AS category_rank
FROM streaming_project.ecommerce.silver_products p
LEFT JOIN streaming_project.ecommerce.gold_popular_products pp
    ON p.product_id = pp.product_id
ORDER BY pp.total_revenue DESC;


3. How is revenue trending month over month?

In [0]:
%sql
-- ================================================
-- MV 3: Monthly Sales Trend
-- How is revenue trending month over month?
-- ================================================
CREATE OR REPLACE MATERIALIZED VIEW streaming_project.ecommerce.mv_monthly_sales_trend
AS
SELECT
    order_year,
    order_month,
    SUM(total_revenue)      AS monthly_revenue,
    SUM(total_units_sold)   AS monthly_units_sold,
    SUM(total_orders)       AS monthly_orders,
    SUM(unique_customers)   AS monthly_unique_customers,
    ROUND(
        SUM(total_revenue) - LAG(SUM(total_revenue))
            OVER (ORDER BY order_year, order_month), 2
    )                       AS revenue_vs_last_month,
    ROUND(
        (SUM(total_revenue) - LAG(SUM(total_revenue))
            OVER (ORDER BY order_year, order_month))
        / NULLIF(LAG(SUM(total_revenue))
            OVER (ORDER BY order_year, order_month), 0) * 100, 2
    )                       AS revenue_growth_pct
FROM streaming_project.ecommerce.gold_category_revenue
GROUP BY order_year, order_month
ORDER BY order_year, order_month;